# 1. Data and features

**Question for the project:** what actually predicts a player's fantasy points *next week*?

This notebook loads five regular seasons (2021-2025) of weekly player stats and Vegas lines from
[nflverse](https://github.com/nflverse/nflverse-data), defines the sample, and builds features that only
use information available **before** kickoff. That last part matters: a feature computed with the
week it's predicting in it would make every model look better than it is.

In [1]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from fantasy.data import load_player_weeks, load_games
from fantasy.features import FEATURES, TARGET, build_features, model_sample, mingus_projection
from fantasy import plots as P

P.use_style()
pd.set_option("display.precision", 3)

In [2]:
players = load_player_weeks()
games = load_games()
print(f"{len(players):,} player-games, {players.player_id.nunique():,} players, seasons {players.season.min()}-{players.season.max()}")
players[["player_display_name", "position", "season", "week", "team", "opponent_team",
         "targets", "carries", "receiving_yards", "rushing_yards", "fantasy_points_ppr"]].sample(5, random_state=1)

29,376 player-games, 1,130 players, seasons 2021-2025


,player_display_name,position,season,week,team,opponent_team,targets,carries,receiving_yards,rushing_yards,fantasy_points_ppr
12183,Scott Miller,WR,2022,8,TB,BAL,2,0,13,0,3.3
27967,Troy Franklin,WR,2025,15,DEN,GB,6,1,85,3,20.8
22678,Trey McBride,TE,2022,17,ARI,ATL,10,0,78,0,20.8
15165,Zack Moss,RB,2021,13,BUF,NE,3,8,12,21,5.3
26650,Jayden Reed,WR,2025,17,GB,BAL,4,0,41,0,8.1


## Scoring check

nflverse ships PPR points already computed. A quick check against a hand calculation
(1 per 25 passing yards, 4 per passing TD, -2 per INT, 1 per 10 rushing/receiving yards, 6 per TD,
1 per reception, -2 per fumble lost, 2 per 2-point conversion) confirms we know what the target means.

In [3]:
hand = (players.passing_yards / 25 + 4 * players.passing_tds - 2 * players.passing_interceptions
        + (players.rushing_yards + players.receiving_yards) / 10
        + 6 * (players.rushing_tds + players.receiving_tds) + players.receptions
        - 2 * players.fumbles_lost + 2 * players.two_pt)
diff = (players.fantasy_points_ppr - hand).abs()
print(f"matches within 0.5 pts: {(diff < 0.5).mean():.1%} of player-games; largest gap {diff.max():.1f}")

matches within 0.5 pts: 99.8% of player-games; largest gap 12.0


## Vegas lines

Each game's over/under and spread turn into an **implied team total**: what the betting market expects
each team to score. Home implied = (total + spread) / 2.

In [4]:
g = games.dropna(subset=["implied_total", "points"])
print(f"implied total vs. actual points: r = {g.implied_total.corr(g.points):.2f}, "
      f"mean absolute miss {(g.implied_total - g.points).abs().mean():.1f} pts")
g[["season", "week", "team", "opponent_team", "total_line", "spread", "implied_total", "points"]].head()

implied total vs. actual points: r = 0.40, mean absolute miss 7.2 pts


,season,week,team,opponent_team,total_line,spread,implied_total,points
0,2021,1,TB,DAL,52.5,10.0,31.25,31.0
1,2021,1,ATL,PHI,49.0,3.5,26.25,6.0
2,2021,1,BUF,PIT,48.0,6.5,27.25,16.0
3,2021,1,CAR,NYJ,44.5,3.5,24.00,19.0
4,2021,1,CIN,MIN,47.0,-3.0,22.00,27.0


## Features

For each player-game, `build_features` adds:

| Feature | Meaning (all from games *before* this week) |
|---|---|
| `last_game`, `avg_last3`, `avg_season` | PPR points last game, last 3 games, season to date |
| `prior_season_avg` | last season's PPR per game (4+ games) |
| `opps_per_game`, `targets_per_game`, `carries_per_game`, `target_share_avg` | volume |
| `tds_per_game` | touchdowns per game (rushing + receiving) |
| `def_factor` | PPR allowed per game by this week's opponent to the position, ÷ league average (1.0 = average) |
| `implied_total`, `implied_factor`, `spread`, `home` | Vegas: team total, total ÷ that week's average, expected margin, home game |

In [5]:
df = build_features(players, games)
sample = model_sample(df)
print(f"typical week: players score {sample[TARGET].sum() / sample.avg_season.sum():.1%} of their season average")
print(f"model sample: {len(sample):,} player-games "
      f"(players with 3+ earlier games this season averaging 5+ PPR)")
sample.groupby("position").agg(rows=(TARGET, "size"), avg_ppr=(TARGET, "mean"), sd_ppr=(TARGET, "std"))

typical week: players score 97.5% of their season average
model sample: 13,107 player-games (players with 3+ earlier games this season averaging 5+ PPR)


,rows,avg_ppr,sd_ppr
position,,,
QB,2043,15.888,8.311
RB,3474,11.480,8.376
TE,2070,9.090,6.982
WR,5520,10.548,8.134


## Leakage check

If the features are built right, `avg_season` for week *t* must equal the mean of the player's
actual points in weeks before *t*. Spot check one player, recomputed by hand:

In [6]:
name = "Ja'Marr Chase"
p = df[(df.player_display_name == name) & (df.season == 2024)][["week", TARGET, "last_game", "avg_last3", "avg_season"]]
p["check"] = p[TARGET].expanding().mean().shift(1)
assert np.allclose(p.avg_season.dropna(), p.check.dropna())
p.head(6)

,week,fantasy_points_ppr,last_game,avg_last3,avg_season,check
19144,1,12.2,NaN,NaN,NaN,NaN
19145,2,7.5,12.2,12.200,12.20,12.20
19146,3,29.8,7.5,9.850,9.85,9.85
19147,4,17.5,29.8,16.500,16.50,16.50
19148,5,41.3,17.5,18.267,16.75,16.75
19149,6,12.2,41.3,29.533,21.66,21.66


In [7]:
df.to_parquet("../data/features.parquet")
print("saved data/features.parquet")

saved data/features.parquet
